## Learning Notebook:  Build a simple RAG workflow using Open-Source tools

### Overview

This tutorial demonstrates how to implement **Retrieval-Augmented Generation (RAG)** using an **open-source Large Language Model (LLM)** and **Pinecone** as a vector database.

### Context

This notebook highlights the most important steps needed to build a simple RAG pipeline using open source tools form Langchain, sentence_transformers, and pincone. 

### Goal:

- **process documents** (TXT, PDF, Word) for RAG
- **embed and store** text in Pinecone
- **retrieve relevant information** from the database
- **generate responses** using an **open-source LLM** (Llama-2, GPT4All, etc.)


Follow the Jupyter notebook below and answer the tasks marked as ✅ Task for Students.


#### **🔧 Step 0: collect some documents**
✅ Task for Students: collect and add some documents in ./doc in the same directory of this notebook. For example, you can download modul descriptions of different study programs in Frankfurt UAS or the slides-decks of Data Mining course.

#### **🔧 Step 1: Install Required Libraries**
✅ Task for Students: try to install the required dependencies on your laptop/PC

#### **📂 Step 2: Load and Prepare Documents (TXT, PDF, Word)**
read documents from a folder (`docs/`) and prepare them for the next steps

In [1]:
import os
import fitz  # PyMuPDF for PDF processing
import docx
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaLLM

# Function to extract text from different document formats
def extract_text_from_file(file_path):
    if file_path.endswith('.txt'):
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read()
    elif file_path.endswith('.pdf'):
        text = ''
        with fitz.open(file_path) as pdf:
            for page in pdf:
                text += page.get_text()
        return text
    elif file_path.endswith('.docx'):
        doc = docx.Document(file_path)
        return '\n'.join([para.text for para in doc.paragraphs])
    else:
        return None  # Unsupported format

In [2]:
# Load documents from the 'docs/' folder
def load_documents(folder_path='docs/'):
    documents = []
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        text = extract_text_from_file(file_path)
        if text:
            documents.append(text)
    return documents

docs = load_documents()

#### **📂 Step 3: chunk the loaded Documents with the RecursiveCharacterTextSplitter from Langchain**
use a chunk size of 500 and chunk overlap of 50

In [3]:
# Split documents into chunks
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

chunks = text_splitter.split_text(' '.join(docs))

print(f'Loaded {len(chunks)} text chunks.')

Loaded 1616 text chunks.


#### **🔎 Step 4: Embed chunks and Store in Pinecone**
We use `sentence-transformers` to convert text into **vector embeddings** and store them in Pinecone.

In [4]:
with open("Pinecone_API_key.txt", "r", encoding="utf-8") as file:
    API_KEY = file.read()

In [5]:
from pinecone import Pinecone, ServerlessSpec

# Initialize Pinecone (Replace 'YOUR_API_KEY' with your Pinecone API key)
pc = Pinecone(api_key=API_KEY)

In [6]:
index_name = "rag-tutorial"
dimension = 384  # Dimension of the model you're using (e.g., all-MiniLM-L6-v2)

# Create index if it doesn't exist
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

# Connect to the index
index = pc.Index(index_name)

In [7]:
from sentence_transformers import SentenceTransformer

# Load embedding model: Loads or creates a SentenceTransformer model that can be used to map sentences / text to embeddings.
embedder = SentenceTransformer("all-MiniLM-L6-v2")  # 384-dim output

In [8]:
# Create vector records for the chunks
vectors = [
    {
        "id": str(i),
        "values": embedder.encode(chunk).tolist(),
        "metadata": {"text": chunk}
    }
    for i, chunk in enumerate(chunks)
]

In [9]:
# Upsert = Update values and metadata if exist, Insert if not. This allows safe re-run of code without creating duplicates.
# Upsert in small batches to avoid payload limit
batch_size = 50
for i in range(0, len(vectors), batch_size):
    batch = vectors[i:i + batch_size]
    index.upsert(batch)

print(f"✅   Inserted {len(vectors)} vectors into '{index_name}'.")

✅   Inserted 1616 vectors into 'rag-tutorial'.


#### **🤖 Step 5: Load an Open-Source LLM for Answer Generation**
OllamaLLM is a LangChain-compatible interface for working with Ollama — a tool that allows you to run large language models (like LLaMA) locally.

In [10]:
from langchain_ollama import OllamaLLM
# The follwoing code creates an instance of the LLM interface, pointing to a local Ollama model called "llama3.2".  
model = OllamaLLM(model="llama3:latest")

#### **🤖 Step 6: write and embedd your question**

In [11]:
question = "Who is the professor of the course Climate Change & Environmental Impacts ?"

# Embed question
question_vector = embedder.encode(question).tolist()

#### **🔄 Step 7: Retrieve Relevant Information & Generate Response**
We retrieve **top-k** relevant documents and use them for answering questions.

In [12]:
# Query Pinecone
results = index.query(
    vector=question_vector,
    top_k=10,
    include_metadata=True
)

In [13]:
#  Prepare retrieved context
contexts = [match["metadata"]["text"] for match in results["matches"]]
context_text = "\n\n".join(contexts)

# Build prompt
prompt = f"""Answer the question based only on the context below.

Context:
{context_text}

Question: {question}
Answer:"""

In [14]:
prompt

'Answer the question based only on the context below.\n\nContext:\nGewichtung nach Leistungspunkten \n10 \nModulbeauftragte/r und hauptamtlich Lehrende \nProf. Dr. Michael Rademacher \n11 \nSonstige Informationen \nSprache: deutsch \nLiteratur: \n- \nClimate Change 2013 - The Physical Science Basis, Contribution of Working Group I to the Fifth Assess-\nment Report of the IPCC, www.ipcc.ch, \n- \nClimate Change 2007 - Impacts, Adaptation and Vulnerability, Contribution of Working Group II to the Fourth\n\nSprache: deutsch \nLiteratur: \n- \nClimate Change 2013 - The Physical Science Basis, Contribution of Working Group I to the Fifth \nAssessment Report of the IPCC, www.ipcc.ch \n- \nIPCC, 2014: Climate Change 2014: Impacts, Adaptation, and Vulnerability. Part A: Global and \nSectoral Aspects. Contribution of Working Group II to the Fifth Assessment Report of the Inter-\ngovernmental Panel on Climate Change [Field, C.B., V.R. Barros, D.J. Dokken, K.J. Mach, M.D.\n\nPanel on Climate Chan

In [15]:
# Query local LLM via Ollama to get an answer
response = model.invoke(prompt)

In [16]:
response

'Based on the context, the answer is:\n\nProf. Dr. Michael Rademacher'